# CSV + API

In this reboot, we are going to use:

- The [Goodreads books](https://www.kaggle.com/jealousleopard/goodreadsbooks) dataset from Kaggle.
- The [Open Library Books API](https://openlibrary.org/dev/docs/api/books)

The goal of this livecode is to load the data from a CSV + loop over rows to enrich each row with information such as:

- List of subjects (Science, Humor, Travel, etc.)
- The cover URL of the book
- Other information you'd find useful in the JSON API

First, download the CSV in the local folder:

In [1]:
!curl -L https://gist.githubusercontent.com/ssaunier/351b17f5a7a009808b60aeacd1f4a036/raw/books.csv > books.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 50 1509k   50  767k    0     0   728k      0  0:00:02  0:00:01  0:00:01  728k
100 1509k  100 1509k    0     0  1124k      0  0:00:01  0:00:01 --:--:-- 1124k


In [2]:
!ls -lh

total 1.6M
-rw-r--r-- 1 meaki 197609  668 Dec 26 13:56 README.md
-rw-r--r-- 1 meaki 197609  21K Dec 26 14:13 Recap.ipynb
-rw-r--r-- 1 meaki 197609 1.5M Dec 26 14:14 books.csv


Then import the usual suspects!

In [3]:
import requests
import pandas as pd
import numpy as np

## Load books from CSV

In [4]:
df = pd.read_csv("books.csv", dtype=str)  # dtype=str: isbn falan bozulmasın
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,# num_pages,ratings_count,text_reviews_count
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,9780439785969,eng,652,1944099,26249
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,9780439358071,eng,870,1996446,27613
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,9780439554930,eng,320,5629932,70390
3,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,9780439554893,eng,352,6267,272
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,9780439655484,eng,435,2149872,33964


Let's add a new column

In [5]:

df["isbn_clean"] = df["isbn13"].fillna(df["isbn"])

df["isbn_clean"] = (
    df["isbn_clean"]
    .astype(str)
    .str.strip()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)
df.loc[df["isbn_clean"].str.lower().isin(["nan", "none", ""]), "isbn_clean"] = np.nan

df["subjects"] = np.nan
df["cover_url"] = np.nan
df["ol_authors"] = np.nan
df["publish_date"] = np.nan
df["number_of_pages"] = np.nan

df[["title", "isbn", "isbn13", "isbn_clean"]].head()


,title,isbn,isbn13,isbn_clean
0,Harry Potter and the Half-Blood Prince (Harry ...,0439785960,9780439785969,9780439785969
1,Harry Potter and the Order of the Phoenix (Har...,0439358078,9780439358071,9780439358071
2,Harry Potter and the Sorcerer's Stone (Harry P...,0439554934,9780439554930,9780439554930
3,Harry Potter and the Chamber of Secrets (Harry...,0439554896,9780439554893,9780439554893
4,Harry Potter and the Prisoner of Azkaban (Harr...,043965548X,9780439655484,9780439655484


## API - Open Library

In [6]:
import requests
from typing import Dict, Any, Optional
import time

session = requests.Session()
session.headers.update({"User-Agent": "workintech-data-library/1.0 (educational)"})

def fetch_openlibrary_one(isbn: str, timeout: int = 20) -> Dict[str, Any]:
    url = "https://openlibrary.org/api/books"
    params = {"bibkeys": f"ISBN:{isbn}", "format": "json", "jscmd": "data"}

    r = session.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()

    key = f"ISBN:{isbn}"
    if key not in data:
        return {}

    b = data[key]

    subjects = [s.get("name") for s in (b.get("subjects", []) or []) if s.get("name")]
    cover = b.get("cover", {}) or {}
    cover_url = cover.get("large") or cover.get("medium") or cover.get("small")
    authors = [a.get("name") for a in (b.get("authors", []) or []) if a.get("name")]

    return {
        "subjects": subjects if subjects else None,
        "cover_url": cover_url,
        "ol_authors": authors if authors else None,
        "publish_date": b.get("publish_date"),
        "number_of_pages": b.get("number_of_pages"),
    }

# quick test
test_isbn = df["isbn_clean"].dropna().iloc[0]
test_isbn, fetch_openlibrary_one(test_isbn)


('9780439785969',
 {'subjects': ['orphans',
   'foster homes',
   'romans',
   'magie',
   'adolescence',
   'Quill Award winner',
   'Scottish Children’s Book Award winner',
   'British Book of the Year Award winner',
   'Fiction',
   'Juvenile fiction',
   'Magic',
   'Schools',
   'Witches',
   'Wizards',
   'New York Times bestseller',
   'Fantasy fiction',
   'nyt:series_books=2006-07-15',
   'nyt:series_books=2006-09-16',
   'Romans, nouvelles, etc. pour la jeunesse',
   'Sorciers',
   'Roman fantastique',
   'Merveilleux',
   'Hogwarts School of Witchcraft and Wizardry (Imaginary place)',
   'Harry Potter (Fictitious character)',
   'Hogwarts School of Witchcraft and Wizardry (Imaginary organization)',
   'Magos',
   'Magia',
   'Ficción juvenil',
   'Escuelas',
   'Novela fantástica',
   'England',
   'School stories',
   'Family',
   'Harry Potter (Fictional character)',
   'Orphans & Foster Homes',
   'Social Themes',
   'Fantasy',
   'Fantasy & Magic',
   'Friendship',
   'R

## Calling the API with multiple ISBNs at a time

In [8]:
import requests

isbn = df["isbn_clean"].dropna().astype(str).iloc[0]

url = "https://openlibrary.org/api/books"
params = {"bibkeys": f"ISBN:{isbn}", "format": "json", "jscmd": "data"}
data = requests.get(url, params=params, timeout=20).json()

b = data.get(f"ISBN:{isbn}", {})
subjects = [s.get("name") for s in b.get("subjects", []) if s.get("name")]
cover = b.get("cover", {})
cover_url = cover.get("large") or cover.get("medium") or cover.get("small")

df.loc[df["isbn_clean"] == isbn, "subjects"] = str(subjects)
df.loc[df["isbn_clean"] == isbn, "cover_url"] = cover_url

print("isbn:", isbn)
print("subjects:", subjects[:10])
print("cover_url:", cover_url)

df.to_csv("books_enriched.csv", index=False)
print("saved: books_enriched.csv")


isbn: 9780439785969
subjects: ['orphans', 'foster homes', 'romans', 'magie', 'adolescence', 'Quill Award winner', 'Scottish Children’s Book Award winner', 'British Book of the Year Award winner', 'Fiction', 'Juvenile fiction']
cover_url: https://covers.openlibrary.org/b/id/14860369-L.jpg
saved: books_enriched.csv
